# 📈 Predictive Analytics & Forecasting

**UIDAI Aadhaar Insights Project**

This notebook implements time-series forecasting to predict future Aadhaar activity.

---

## 📌 Techniques Covered
1. **Time-Series Decomposition** - Trend, Seasonality, Residuals
2. **Moving Average Forecast** - Simple baseline
3. **Exponential Smoothing** - Weighted forecast
4. **ARIMA/SARIMA** - Advanced time-series modeling
5. **Model Evaluation** - RMSE, MAE, MAPE

In [ ]:
!pip install statsmodels

In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from pathlib import Path
import warnings

warnings.filterwarnings('ignore')

# Time-series libraries
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from sklearn.metrics import mean_squared_error, mean_absolute_error

# Try importing ARIMA
try:
    from statsmodels.tsa.arima.model import ARIMA
    ARIMA_AVAILABLE = True
except ImportError:
    ARIMA_AVAILABLE = False
    print('⚠️ ARIMA not available. Install statsmodels: pip install statsmodels')

# Define paths
DATA_DIR = Path('../data/processed')

print('✅ Libraries loaded!')

In [ ]:
# Load and prepare data
enrolment = pd.read_csv(DATA_DIR / 'cleaned_enrolment.csv', parse_dates=['date'])

# Calculate daily totals
numeric_cols = enrolment.select_dtypes(include=[np.number]).columns.tolist()
numeric_cols = [c for c in numeric_cols if 'pincode' not in c.lower()]
enrolment['total'] = enrolment[numeric_cols].sum(axis=1)

# Aggregate to daily
daily = enrolment.groupby('date')['total'].sum().reset_index()
daily = daily.sort_values('date').set_index('date')

print(f'✅ Daily time series created: {len(daily)} days')
print(f'   Date range: {daily.index.min()} to {daily.index.max()}')

---

## 1️⃣ Time-Series Decomposition

Break down the series into:
- **Trend**: Long-term direction
- **Seasonality**: Repeating patterns
- **Residuals**: Random noise

In [ ]:
# Resample to weekly for cleaner decomposition
weekly = daily.resample('W').sum()

# Perform decomposition
if len(weekly) >= 4:  # Need at least 2 periods
    decomposition = seasonal_decompose(weekly['total'], model='additive', period=4)
    
    fig = make_subplots(rows=4, cols=1, shared_xaxes=True,
                        subplot_titles=['Original', 'Trend', 'Seasonal', 'Residual'])
    
    fig.add_trace(go.Scatter(x=weekly.index, y=weekly['total'], name='Original', line=dict(color='steelblue')), row=1, col=1)
    fig.add_trace(go.Scatter(x=weekly.index, y=decomposition.trend, name='Trend', line=dict(color='green')), row=2, col=1)
    fig.add_trace(go.Scatter(x=weekly.index, y=decomposition.seasonal, name='Seasonal', line=dict(color='orange')), row=3, col=1)
    fig.add_trace(go.Scatter(x=weekly.index, y=decomposition.resid, name='Residual', line=dict(color='gray')), row=4, col=1)
    
    fig.update_layout(height=700, title_text='📊 Time-Series Decomposition (Weekly)', showlegend=False)
    fig.show()
else:
    print('⚠️ Not enough data points for decomposition')

---

## 2️⃣ Train/Test Split

Split data for model evaluation.

In [ ]:
# Use 80% for training, 20% for testing
train_size = int(len(daily) * 0.8)
train = daily.iloc[:train_size]
test = daily.iloc[train_size:]

print(f'✅ Data split:')
print(f'   Train: {len(train)} days ({train.index.min()} to {train.index.max()})')
print(f'   Test: {len(test)} days ({test.index.min()} to {test.index.max()})')

In [ ]:
# Visualize split
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=train.index, y=train['total'],
    mode='lines', name='Training Data',
    line=dict(color='steelblue')
))
fig.add_trace(go.Scatter(
    x=test.index, y=test['total'],
    mode='lines', name='Test Data',
    line=dict(color='orange')
))

fig.add_vline(x=train.index.max(), line_dash='dash', line_color='red',
              annotation_text='Train/Test Split')

fig.update_layout(
    title='📊 Train/Test Split',
    xaxis_title='Date',
    yaxis_title='Total Enrolments',
    height=400
)
fig.show()

---

## 3️⃣ Baseline Forecast: Moving Average

In [ ]:
# Simple Moving Average
window = 7
ma_prediction = train['total'].rolling(window=window).mean().iloc[-1]
ma_forecast = pd.Series([ma_prediction] * len(test), index=test.index)

# Calculate metrics
ma_rmse = np.sqrt(mean_squared_error(test['total'], ma_forecast))
ma_mae = mean_absolute_error(test['total'], ma_forecast)

print(f'📊 Moving Average Baseline (window={window}):')
print(f'   Prediction: {ma_prediction:,.0f}/day')
print(f'   RMSE: {ma_rmse:,.2f}')
print(f'   MAE: {ma_mae:,.2f}')

---

## 4️⃣ Exponential Smoothing

In [ ]:
# Fit Exponential Smoothing model
try:
    es_model = ExponentialSmoothing(
        train['total'],
        trend='add',
        seasonal=None,
        initialization_method='estimated'
    )
    es_fit = es_model.fit()
    es_forecast = es_fit.forecast(len(test))
    es_forecast.index = test.index
    
    # Calculate metrics
    es_rmse = np.sqrt(mean_squared_error(test['total'], es_forecast))
    es_mae = mean_absolute_error(test['total'], es_forecast)
    
    print(f'📊 Exponential Smoothing:')
    print(f'   RMSE: {es_rmse:,.2f}')
    print(f'   MAE: {es_mae:,.2f}')
    
    ES_AVAILABLE = True
except Exception as e:
    print(f'⚠️ Exponential Smoothing failed: {e}')
    ES_AVAILABLE = False

---

## 5️⃣ ARIMA Forecasting

In [ ]:
if ARIMA_AVAILABLE:
    try:
        # Fit ARIMA model
        arima_model = ARIMA(train['total'], order=(5, 1, 2))
        arima_fit = arima_model.fit()
        
        # Forecast
        arima_forecast = arima_fit.forecast(steps=len(test))
        arima_forecast.index = test.index
        
        # Calculate metrics
        arima_rmse = np.sqrt(mean_squared_error(test['total'], arima_forecast))
        arima_mae = mean_absolute_error(test['total'], arima_forecast)
        
        print(f'📊 ARIMA(5,1,2) Model:')
        print(f'   RMSE: {arima_rmse:,.2f}')
        print(f'   MAE: {arima_mae:,.2f}')
        
        ARIMA_SUCCESS = True
    except Exception as e:
        print(f'⚠️ ARIMA failed: {e}')
        ARIMA_SUCCESS = False
else:
    ARIMA_SUCCESS = False
    print('⚠️ ARIMA not available')

---

## 6️⃣ Model Comparison

In [ ]:
# Compare all models
fig = go.Figure()

# Actual test data
fig.add_trace(go.Scatter(
    x=test.index, y=test['total'],
    mode='lines', name='Actual',
    line=dict(color='black', width=2)
))

# Moving Average
fig.add_trace(go.Scatter(
    x=ma_forecast.index, y=ma_forecast.values,
    mode='lines', name='Moving Average',
    line=dict(color='gray', dash='dash')
))

# Exponential Smoothing
if ES_AVAILABLE:
    fig.add_trace(go.Scatter(
        x=es_forecast.index, y=es_forecast.values,
        mode='lines', name='Exp. Smoothing',
        line=dict(color='green')
    ))

# ARIMA
if ARIMA_SUCCESS:
    fig.add_trace(go.Scatter(
        x=arima_forecast.index, y=arima_forecast.values,
        mode='lines', name='ARIMA',
        line=dict(color='red')
    ))

fig.update_layout(
    title='📈 Forecast Comparison: Actual vs Predicted',
    xaxis_title='Date',
    yaxis_title='Total Enrolments',
    height=450,
    hovermode='x unified'
)
fig.show()

In [ ]:
# Model comparison table
results = []

results.append({'Model': 'Moving Average', 'RMSE': ma_rmse, 'MAE': ma_mae})

if ES_AVAILABLE:
    results.append({'Model': 'Exponential Smoothing', 'RMSE': es_rmse, 'MAE': es_mae})

if ARIMA_SUCCESS:
    results.append({'Model': 'ARIMA(5,1,2)', 'RMSE': arima_rmse, 'MAE': arima_mae})

results_df = pd.DataFrame(results).sort_values('RMSE')

print('=' * 60)
print('📊 MODEL COMPARISON')
print('=' * 60)
print(results_df.to_string(index=False))
print(f'\n🏆 Best Model: {results_df.iloc[0]["Model"]}')

---

## 7️⃣ Future Forecast (Next 30 Days)

In [ ]:
# Generate future forecast using best available model
forecast_days = 30

# Fit on all data
if ARIMA_AVAILABLE:
    try:
        full_model = ARIMA(daily['total'], order=(5, 1, 2))
        full_fit = full_model.fit()
        future_forecast = full_fit.forecast(steps=forecast_days)
        
        # Create future dates
        last_date = daily.index.max()
        future_dates = pd.date_range(start=last_date + pd.Timedelta(days=1), periods=forecast_days)
        future_forecast.index = future_dates
        
        FORECAST_SUCCESS = True
    except:
        FORECAST_SUCCESS = False
else:
    FORECAST_SUCCESS = False

# Fallback to exponential smoothing
if not FORECAST_SUCCESS:
    try:
        full_model = ExponentialSmoothing(daily['total'], trend='add', initialization_method='estimated')
        full_fit = full_model.fit()
        future_forecast = full_fit.forecast(forecast_days)
        
        last_date = daily.index.max()
        future_dates = pd.date_range(start=last_date + pd.Timedelta(days=1), periods=forecast_days)
        future_forecast.index = future_dates
        
        FORECAST_SUCCESS = True
    except:
        FORECAST_SUCCESS = False

In [ ]:
if FORECAST_SUCCESS:
    # Visualize future forecast
    fig = go.Figure()
    
    # Historical data (last 60 days)
    recent = daily.iloc[-60:]
    fig.add_trace(go.Scatter(
        x=recent.index, y=recent['total'],
        mode='lines', name='Historical',
        line=dict(color='steelblue', width=2)
    ))
    
    # Forecast
    fig.add_trace(go.Scatter(
        x=future_forecast.index, y=future_forecast.values,
        mode='lines+markers', name='Forecast',
        line=dict(color='green', width=2, dash='dash')
    ))
    
    fig.add_vline(x=daily.index.max(), line_dash='dot', line_color='red',
                  annotation_text='Forecast Start')
    
    fig.update_layout(
        title=f'🔮 {forecast_days}-Day Forecast',
        xaxis_title='Date',
        yaxis_title='Predicted Enrolments',
        height=450
    )
    fig.show()
    
    # Print forecast summary
    print('=' * 60)
    print(f'🔮 {forecast_days}-DAY FORECAST SUMMARY')
    print('=' * 60)
    print(f'   Predicted Daily Average: {future_forecast.mean():,.0f}')
    print(f'   Predicted Total: {future_forecast.sum():,.0f}')
    print(f'   Min Day: {future_forecast.min():,.0f}')
    print(f'   Max Day: {future_forecast.max():,.0f}')
else:
    print('⚠️ Could not generate forecast')

---

## 📌 Key Findings

### Time-Series Patterns:
- Data shows clear trend and potential seasonality
- Weekly patterns may influence daily variations

### Model Performance:
- Multiple models tested for comparison
- Best model selected based on RMSE/MAE

### Forecast Insights:
- Predictions generated for next 30 days
- Can be used for capacity planning

